# 14 — Executive Assistant
> A raw email thread or meeting transcript goes in — a ready-to-send reply, a prioritised task list, and a follow-up tracker come back together in a single pass.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


class ActionItem(BaseModel):
    description: str = Field(description="What needs to be done")
    owner: Optional[str] = Field(default=None, description="Person responsible")
    due_date: Optional[str] = Field(default=None, description="Deadline, e.g. 'Friday 20 Jun'")
    priority: Literal["high", "medium", "low"]


class FollowUpEntry(BaseModel):
    topic: str = Field(description="What is being tracked")
    waiting_on: Optional[str] = Field(default=None, description="Who must act next")
    check_in_by: Optional[str] = Field(default=None, description="When to chase if no update")
    notes: str = Field(description="Context needed to follow up effectively")


class ExecOutput(BaseModel):
    """Structured executive assistant output from an email thread or meeting transcript."""

    input_type: Literal["email_thread", "meeting_transcript"]
    draft_reply: str = Field(
        description="A polished draft reply or acknowledgement ready to send"
    )
    subject_line: Optional[str] = Field(
        default=None,
        description="Suggested email subject line (email_thread inputs only)",
    )
    action_items: List[ActionItem] = Field(
        description="All discrete tasks extracted, with owner and deadline where identifiable"
    )
    follow_up_tracker: List[FollowUpEntry] = Field(
        description="Items that need monitoring but are not direct action items yet"
    )
    meeting_summary: Optional[str] = Field(
        default=None,
        description="2-4 sentence summary (meeting_transcript inputs only)",
    )


ASSISTANT_SYSTEM = SystemMessage(
    """You are a world-class executive assistant.

From the input (either an email thread or a meeting transcript) produce ALL of the
following in one pass:

1. draft_reply      -- A polished, ready-to-send reply or acknowledgement.
                       If there is nothing substantive to reply to, draft a brief
                       acknowledgement that shows the email was read and understood.
2. action_items     -- Every discrete task mentioned or implied. Name the owner and
                       deadline where the text makes them identifiable. Priority:
                       "high" = blocking or time-sensitive, "medium" = this week,
                       "low" = nice-to-have.
3. follow_up_tracker -- Items that need monitoring but are not direct tasks yet
                       (e.g. "waiting for legal to respond", "call scheduled TBC").
4. meeting_summary  -- Populate only if the input is a meeting transcript.
5. subject_line     -- Populate only if the input is an email thread.

Set input_type = "email_thread" or "meeting_transcript" based on the content.
Every output must be populated -- no empty lists."""
)


def run(input_text: str) -> ExecOutput:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    assistant = ASSISTANT_SYSTEM | llm.with_structured_output(ExecOutput)
    return assistant.invoke(
        HumanMessage(content="Input to process:\n\n" + input_text)
    )

## The scenario

A three-way email thread between a CFO, the Head of Finance, and an external auditor. The audit firm needs three documents before Friday to issue a management letter in time for Monday's board meeting — two of those are ready internally but one is still stuck with Legal on a lease dispute. The agent will draft a holding reply that buys time without burning the relationship, extract every commitment made across the thread, and flag what is still waiting on a third party.

In [ ]:
SAMPLE_EMAIL_THREAD = """
From: Martin Köhler <m.kohler@pinnacleaudit.com>
To: Rachel Voss <r.voss@meridiangroup.com>
Cc: Tom Huang <t.huang@meridiangroup.com>
Subject: RE: Q2 Audit — Outstanding Items
Date: Wednesday 18 June 2025, 08:45

Rachel,

Following our call last Thursday, we still require the following before
we can sign off on the Q2 accounts:

1. Bank reconciliation statements for April and May (both entities).
2. Board-approved capex schedule for H1 2025.
3. Written confirmation from your legal team on the status of the
   São Paulo lease dispute — our file shows this as unresolved.

Our deadline to complete fieldwork is Friday 20 June. If we do not
receive items 1 and 2 by COB Thursday, we will not be able to issue
the management letter in time for your Monday board meeting.

Martin Köhler | Senior Manager, Pinnacle Audit

---

From: Tom Huang <t.huang@meridiangroup.com>
To: Rachel Voss <r.voss@meridiangroup.com>
Subject: RE: Q2 Audit — Outstanding Items
Date: Wednesday 18 June 2025, 10:20

Rachel,

Bank recs are ready — I can send them today. The capex schedule needs
your sign-off before I can release it to Martin. The São Paulo item is
still with Legal (Maria's team) — last I heard they were waiting on
the local counsel report which was expected end of last week.

Let me know how you want to handle the board presentation gap if the
letter isn't ready in time.

Tom
"""

result = run(SAMPLE_EMAIL_THREAD)

print("=" * 65)
print(f"EXEC ASSISTANT OUTPUT | Type: {result.input_type}")
print("=" * 65)

if result.subject_line:
    print(f"\nSubject: {result.subject_line}")

print("\nDRAFT REPLY:")
print(result.draft_reply)

print(f"\nACTION ITEMS ({len(result.action_items)}):")
for i, a in enumerate(result.action_items, 1):
    owner = a.owner or "TBC"
    due = a.due_date or "TBC"
    print(f"  {i}. [{a.priority.upper()}] {a.description}")
    print(f"     Owner: {owner} | Due: {due}")

print(f"\nFOLLOW-UP TRACKER ({len(result.follow_up_tracker)}):")
for f in result.follow_up_tracker:
    waiting = f.waiting_on or "TBC"
    check = f.check_in_by or "TBC"
    print(f"  - {f.topic}")
    print(f"    Waiting on: {waiting} | Check in by: {check}")
    print(f"    Notes: {f.notes}")

## Use your own data

Replace the input above with:
- your email thread (paste the full chain, headers included)
- or a meeting transcript (copy-paste from Zoom, Teams, or Otter)

The agent will return a ready-to-send reply, every action item with owner and deadline, and a tracker of anything still waiting on someone else.